In [1]:
## RUN THIS FIRST WHEN ON COLAB (uncomment the lines below). Skip entirely when running locally.
# Google Drive is mounted in the setup cell further down, so no need to mount it here.

# Clone the repository
!git clone https://github.com/benpsutton/water_hyacinth_code.git

# Change into the repo
%cd /content/water_hyacinth_code/

# Install extra packages (if needed)
!pip install torchmetrics kornia optuna

Cloning into 'water_hyacinth_code'...
remote: Enumerating objects: 575, done.
remote: Counting objects: 100% (575/575), done.
remote: Compressing objects: 100% (336/336), done.
remote: Total 575 (delta 360), reused 433 (delta 227), pack-reused 0 (from 0)
Receiving objects: 100% (575/575), 12.23 MiB | 20.29 MiB/s, done.
Resolving deltas: 100% (360/360), done.
/content/water_hyacinth_code
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 58.7 MB/s eta 0:00:00


In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchmetrics
import kornia.augmentation as K
import pandas as pd
from statistics import median
import optuna
import sys

In [3]:
# --- Environment detection & paths ---
# Runs unchanged on both local (VSCode/Jupyter) and Colab.
#   PROJECT_ROOT -> where code + input data live (read from)
#   OUTPUT_ROOT  -> where results are saved (written to). On Colab this is Google Drive,
#                   so plots/CSVs/models persist after the runtime disconnects.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/water_hyacinth_code")                  # the cloned repo
    OUTPUT_ROOT = Path("/content/drive/MyDrive/Dissertation")    # persists in Drive
else:
    PROJECT_ROOT = Path().resolve().parent                              # notebooks/ -> repo root
    OUTPUT_ROOT = PROJECT_ROOT                                          # save in-repo, as before

# Create output dirs up front (plot_loss_curve / to_csv don't make them themselves)
(OUTPUT_ROOT / "outputs" / "plots" / "loss_curves").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "outputs" / "CNN_outputs" / "models").mkdir(parents=True, exist_ok=True)

datetime_now = datetime.datetime.now()
date = datetime_now.strftime("%Y-%m-%d")

SEED = 42
torch.manual_seed(SEED)

print(f"IN_COLAB:     {IN_COLAB}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"OUTPUT_ROOT:  {OUTPUT_ROOT}")

Mounted at /content/drive
IN_COLAB:     True
PROJECT_ROOT: /content/water_hyacinth_code
OUTPUT_ROOT:  /content/drive/MyDrive/Dissertation


In [4]:
sys.path.append(str(PROJECT_ROOT / "src"))

from model_utils import load_patch_dict_as_tensor, buildCNN_2xVGG, predict_on_test_region, train_for_one_epoch, val_for_one_epoch, load_dataset, record_epoch, record_inner_loop_best_state, record_outer_loop_metrics
from plotting import plot_loss_curve

In [5]:
# Load patches with specified size

patch_size = 15

patches_path = OUTPUT_ROOT / f"outputs/{patch_size}px_patches" / f"combined_{patch_size}px_patches.geojson"

with open(patches_path, "r") as f:
    patches_dict = json.load(f)


In [6]:

# # --- Device selection ---
# # Colab (with a GPU runtime: Runtime > Change runtime type > GPU):
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Apple Silicon (M1/M2/M3/M4), using the Metal Performance Shaders backend:
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Portable version that works on either without editing — checks cuda, then mps, then falls back to cpu:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

aug = K.AugmentationSequential(
    K.RandomRotation(degrees = 90.0, p =0.5),
    K.RandomHorizontalFlip(p= 0.25),
    K.RandomVerticalFlip(p= 0.25),
    data_keys= ['input']).to(device)

# metrics - use torchmetrics to make a metric collection that is a dictionary of metrics

metrics = torchmetrics.MetricCollection({
    "accuracy": torchmetrics.classification.BinaryAccuracy(),
    "f1": torchmetrics.classification.BinaryF1Score(),
    "precision": torchmetrics.classification.BinaryPrecision(),
    "recall": torchmetrics.classification.BinaryRecall(),
    "auroc": torchmetrics.classification.BinaryAUROC()
}).to(device)


# Nested loop without hyperparameter tuning

In [ ]:
# # Create a nested loop without doing hyperparameter tuning

# band_list = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

# patches, labels, locations, class_int, label_id = load_patch_dict_as_tensor(patches_dict, band_list= band_list)
# del patches_dict

# input_channels = len(band_list)

# num_epochs = 2
# best_val_loss = float("inf")
# patience = 10
# epochs_no_improve = 0
# best_state = None
# best_val_metrics = None
# best_epoch = None


# list_of_run_histories = [] # will store per-epoch training and validation metrics of EVERY run for later plotting
# list_of_predicition_dfs = [] # will store predictions and labels for each of the outer fold test regions for confusion matrix
# list_of_inner_best_state_metrics = [] # will store the training vand validation losses and other metrics for the best model for each inner fold.

# outer_metrics_dict= {"run_ID": [], "patch_size": [], "test_region": [], "epochs_trained": [], "test_loss": [], "test_accuracy": [], "test_f1": [], "test_precision": [],
#                      "test_recall": [], "test_auroc": [], "train_loss": [], "train_accuracy": [], "train_f1": [], "train_precision": [], "train_recall": [], "train_auroc": []}
# # will store final epoch training loss, and testing loss and other metrics for the outer folds

# location_list = ["Hartbeespoort", "Rodman", "Mula", "Inle", "Vembanad", "Valsequillo", "RawaPening", "Winam"]

# for location in location_list:

#     test_region = location
#     remaining_regions = [loc for loc in location_list if loc != location]
#     test_abrv = test_region[0:3]
#     # print(test_abrv)

#     # Need a fresh best_state_dict for each outer fold so can get final metrics of the inner fold runs.
#     inner_best_state_dict= {"run_ID": [], "patch_size":[], "test_region": [], "val_region": [], "train_loss": [], "val_loss": [], "epoch_stopped": [],
#                   "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": [], "val_auroc":[], }

#     for region in remaining_regions:

#         val_region = region
#         training_regions = [loc for loc in remaining_regions if loc != region]
#         val_abrv = val_region[0:3]

#         run_ID = f"T-{test_abrv}_V-{val_abrv}_{patch_size}px_{date}"


#         model = buildCNN_2xVGG(input_channels= input_channels, dropout = 0.3)
#         model = model.to(device)
#         criterion = nn.BCEWithLogitsLoss()
#         optimizer = optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-4)
#         # was: scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
#         scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5) # reduces the step size by half every 5 epochs/

#         num_epochs = 100
#         best_val_loss = float("inf")
#         patience = 10
#         epochs_no_improve = 0
#         best_state = None
#         best_val_metrics = None
#         best_epoch = None


#         # Create history_dict for each run, with values for each epoch, that will then be contanated into a long dataframe for plotting loss_curves / analysis
#         history_dict = {"test_region": [], "val_region": [],"run_ID": [], "patch_size":[], "loop":[], "epoch":[], "train_loss": [], "val_loss": [], "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": [], "val_auroc":[],
#                         "train_accuracy":[],"train_f1":[], "train_precision":[], "train_recall":[], "train_auroc":[]
#                         }

#         # train_data, val_data, train laoder, val loader
#         #-------------------------------------------------------
#         test_mask = locations == test_region
#         val_mask = locations == val_region

#         train_mask = ~test_mask & ~ val_mask

#         train_patches, train_labels = patches[train_mask], labels[train_mask]
#         val_patches, val_labels = patches[val_mask], labels[val_mask]
#         # test_patches, test_labels, test_labels_ids = patches[test_mask], labels[test_mask], label_id[test_mask]


#         mean = train_patches.mean(dim= (0,2,3)) # will average over the 0,2,3 axes and keep the 1 axis (channels / bands) separate. output shape (C,)
#         std = train_patches.std(dim = (0,2,3))

#         train_loader = load_dataset(patches=train_patches,
#                                     labels= train_labels,
#                                     mean= mean,
#                                     std = std,
#                                     batch_size= 32,
#                                     shuffle= True)

#         val_loader = load_dataset(patches = val_patches,
#                                   labels= val_labels,
#                                   mean= mean,
#                                   std= std,
#                                   shuffle = False,
#                                   batch_size= 32
#                                   )

#         for epoch in range(num_epochs):

#             train_loss,train_metrics = train_for_one_epoch(model, train_loader, device, aug, optimizer, metrics, criterion)
#             val_loss, val_metrics = val_for_one_epoch(model, val_loader, metrics, device, criterion)

#             # Save metrics for epoch to history dict
#             record_epoch(history_dict= history_dict, run_ID=run_ID, patch_size = patch_size, loop = "inner", epoch=epoch, test_region=test_region, val_region=val_region,
#                          train_loss=train_loss, val_loss= val_loss, train_metrics= train_metrics, val_metrics=val_metrics )

#             scheduler.step()

#             print(f"Epoch {epoch+1}/100 | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} "
#                 f"| val_f1: {val_metrics['f1']:.4f} | val_acc: {val_metrics['accuracy']:.4f}")

#             # Early stopping

#             if val_loss < best_val_loss:
#                 best_val_loss = val_loss
#                 best_train_loss = train_loss
#                 best_val_metrics = val_metrics
#                 best_epoch = epoch
#                 epochs_no_improve = 0
#                 best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
#             else:
#                 epochs_no_improve +=1
#                 if epochs_no_improve >= patience:
#                     print(f"Early stopping at epcoch: {epoch+1}")
#                     break

#         # model.load_state_dict(best_state)

#         loss_curve_fp = OUTPUT_ROOT / "outputs/plots/loss_curves" / f"{run_ID}.png"
#         plot_loss_curve(history_dict, loss_curve_fp)

#         print(run_ID)
#         print(f"Best epoch: {best_epoch+1} | val_loss: {best_val_loss:.4f} | val_metrics: {best_val_metrics}")

#         # Append this run's history dict to list for later concatenation.
#         list_of_run_histories.append(history_dict)

#         # Save metrics for the best state after training / early stopping

#         record_inner_loop_best_state(inner_best_state_dict, run_ID, patch_size, best_epoch, test_region, val_region, train_loss, best_val_loss, best_val_metrics)

#     list_of_inner_best_state_metrics.append(inner_best_state_dict)

#     # -------------------------
#     # Outer fold -> train on all remaining regions using median number of epochs from inner fold
#     #---------------------------------------------------------------------------------------------
#     history_dict = {"run_ID": [], "test_region": [], "patch_size":[], "loop":[], "epoch":[], "train_loss": [], "train_accuracy":[],"train_f1":[], "train_precision":[], "train_recall":[], "train_auroc":[]
#                         }

#     test_mask = locations == test_region
#     remaining_mask = locations != test_region

#     run_ID = f"Test_{test_abrv}_outer_{patch_size}px_{date}"

#     remaining_patches = patches[remaining_mask]
#     remaining_mean = remaining_patches.mean(dim= (0,2,3))
#     remaining_std =  remaining_patches.std(dim= (0,2,3))

#     test_patches = patches[test_mask]

#     remaining_labels = labels[remaining_mask]
#     test_labels = labels[test_mask]
#     test_label_ids = label_id[test_mask]
#     test_class_int = class_int[test_mask]

#     remaining_loader = load_dataset(patches=remaining_patches,
#                                     labels= remaining_labels,
#                                     mean= remaining_mean,
#                                     std= remaining_std,
#                                     shuffle = True,
#                                     batch_size = 32
#                                     )


#     test_loader = load_dataset(patches=test_patches,
#                                labels = test_labels,
#                                mean = remaining_mean,
#                                std = remaining_std,
#                                shuffle= False,
#                                batch_size= 32
#                                )

#     model = buildCNN_2xVGG(input_channels= input_channels, dropout = 0.3)
#     model = model.to(device)
#     criterion = nn.BCEWithLogitsLoss()
#     optimizer = optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-4)

#     scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

#     epochs_stopped = list(inner_best_state_dict["epoch_stopped"])
#     median_epoch = int(median(epochs_stopped))

#     for epoch in range(median_epoch):
#         train_loss, train_metrics = train_for_one_epoch(model, remaining_loader, device, aug, optimizer, metrics, criterion)

#         scheduler.step()

#         print(f"Epoch {epoch+1}/100 | train_loss: {train_loss:.4f}")

#         # save the per epoch metrics so can plot loss curve later

#         record_epoch(history_dict= history_dict, run_ID=run_ID, patch_size = patch_size, loop = "outer", epoch=epoch, test_region=test_region,
#                          train_loss=train_loss, train_metrics= train_metrics)

#         loss_curve_fp = OUTPUT_ROOT / "outputs/plots/loss_curves" / f"{run_ID}.png"

#     # Append trainining history dict
#     list_of_run_histories.append(history_dict)

#     # Use model to predict on the held-out test region, save preds for creating confusion matrix.

#     test_loss, test_metrics, preds_tensor, logits = predict_on_test_region(model, test_loader, metrics, device, criterion)

#     loss_curve_fp = OUTPUT_ROOT / "outputs/plots/loss_curves" / f"{run_ID}.png"

#     plot_loss_curve(history_dict, loss_curve_fp)

#     record_outer_loop_mterics(outer_metrics_dict, run_ID, patch_size, test_region, median_epoch, test_loss, train_loss, test_metrics, train_metrics)


# # Need to put together the preds with labels etc

#     preds_array = preds_tensor.detach().squeeze(1).cpu().numpy()
#     labels_array = test_labels.detach().squeeze(1).numpy()

#     prediction_df = pd.DataFrame({
#         "labels": labels_array,
#         "predicted": preds_array,
#         "class_int": test_class_int,
#         "test_region": test_region,
#         "run_id": run_ID
#     })

#     list_of_predicition_dfs.append(prediction_df)

#     # save trained models and the relevant parameters for each outer fold?

#     checkpoint = {
#         "run_ID": run_ID,
#         "test_region": test_region,
#         "model_state_dict": model.state_dict(),
#         "mean": remaining_mean,
#         "std": remaining_std,
#         "band_list": band_list,
#         "median_epoch": median_epoch,
#         "hyperparams": {"lr": 1e-3, "weight_decay": 1e-4, "dropout": 0.3, "batch_size": 32},
#     }
#     model_fp = OUTPUT_ROOT / "outputs" / "CNN_outputs" / "models" / f"{run_ID}.pt"
#     model_fp.parent.mkdir(parents=True, exist_ok=True)
#     torch.save(checkpoint, model_fp)

# #----------------------------------------------------------------------------------
# # After loops
# # Concat the the prediciton_dfs

# predictions = pd.concat(list_of_predicition_dfs,axis=0)

# # Need to combine the history dicts into a data frame
# history_df = pd.concat([pd.DataFrame(h) for h in list_of_run_histories], ignore_index= True)

# run_history_fp = OUTPUT_ROOT / "outputs" / "CNN_outputs" / f"training_run_history_{patch_size}px_{date}.csv"
# history_df.to_csv(run_history_fp)

# # save the outer metrics as data frame

# outer_metrics_df =pd.DataFrame(outer_metrics_dict)

# outer_metrics_fp = OUTPUT_ROOT / "outputs" / "CNN_outputs" / f"Outer_fold_metrics_{patch_size}px_{date}.csv"

# outer_metrics_df.to_csv(outer_metrics_fp)
# # save the best inner metrics as dataframe for each fold

# inner_metrics_df = pd.concat([pd.DataFrame(h) for h in list_of_inner_best_state_metrics], ignore_index= True)

# inner_metrics_fp = OUTPUT_ROOT / "outputs" / "CNN_outputs" / f"Inner_fold_best_state_metrics_{patch_size}px_{date}.csv"
# inner_metrics_df.to_csv(inner_metrics_fp)



In [7]:
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# Nested Loop with Optuna hyperparameter tuning on the inner fold

In [9]:
# Define an objective function in the outer loop which wraps the inner loop

band_list = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]

patches, labels, locations, class_int, label_id = load_patch_dict_as_tensor(patches_dict, band_list= band_list)
del patches_dict

input_channels = len(band_list)

num_epochs = 100
best_val_loss = float("inf")
patience = 10
epochs_no_improve = 0
best_state = None
best_val_metrics = None
best_epoch = None
batch_size = 64

list_of_run_histories = [] # will store per-epoch training and validation metrics of all best trial and test runs for later plotting
list_of_predicition_dfs = [] # will store predictions and labels for each of the outer fold test regions for confusion matrix
list_of_inner_best_state_metrics = [] # will store the training vand validation losses and other metrics for the best model for each inner fold.

outer_metrics_dict= {"run_ID": [], "patch_size": [], "test_region": [], "epochs_trained": [], "test_loss": [], "test_accuracy": [], "test_f1": [], "test_precision": [],
                     "test_recall": [], "test_auroc": [], "train_loss": [], "train_accuracy": [], "train_f1": [], "train_precision": [], "train_recall": [], "train_auroc": [],
                     "dropout": [], "lr": [], "weight_decay":[], "batch_size": []}
# will store final epoch training loss, and testing loss and other metrics for the outer folds

location_list = ["Hartbeespoort", "Rodman", "Mula", "Inle", "Vembanad", "Valsequillo", "RawaPening", "Winam"]

for location in location_list:

    test_region = location
    remaining_regions = [loc for loc in location_list if loc != location]
    test_abrv = test_region[0:3]
    # print(test_abrv)



    def objective(trial):

        # Learning rate
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log = True)
        # dropout
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        # weight decay
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log = True)

        inner_val_losses = [] # by optuna to asses perfmance at each trial
        inner_f1_scores = [] # If want to select optuna trials based on f1 score instead
        inner_best_epochs = [] # which epoch numbers is best for each run in the trial -> use mean to set epochs for training on test
        fold_metrics = [] # will be added to the user attributes for each trial, then can easity access the metrics of the best_trial
        fold_histories = [] # to append the training histry_dict for each fold in the trial to allow plotting of the loss curve for thr best trial

        for region in remaining_regions:

            val_region = region
            training_regions = [loc for loc in remaining_regions if loc != region]
            val_abrv = val_region[0:3]

            run_ID = f"T-{test_abrv}_V-{val_abrv}_{patch_size}px_{date}"

            model = buildCNN_2xVGG(input_channels= input_channels, dropout = dropout)
            model = model.to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = optim.Adam(model.parameters(), lr = lr, weight_decay = weight_decay)
            # was: scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5) # reduces the step size by half every 5 epochs/

            num_epochs = 100
            best_val_loss = float("inf")
            patience = 10
            epochs_no_improve = 0
            best_state = None
            best_val_metrics = None
            best_epoch = None
            best_train_metrics = None


            # Create history_dict for each run, with values for each epoch, that will then be contanated into a long dataframe for plotting loss_curves / analysis
            history_dict = {"test_region": [], "val_region": [],"run_ID": [], "patch_size":[], "loop":[], "epoch":[], "train_loss": [], "val_loss": [], "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": [], "val_auroc":[],
                            "train_accuracy":[],"train_f1":[], "train_precision":[], "train_recall":[], "train_auroc":[]
                            }

            # train_data, val_data, train laoder, val loader
            #-------------------------------------------------------
            test_mask = locations == test_region
            val_mask = locations == val_region

            train_mask = ~test_mask & ~ val_mask

            train_patches, train_labels = patches[train_mask], labels[train_mask]
            val_patches, val_labels = patches[val_mask], labels[val_mask]
            # test_patches, test_labels, test_labels_ids = patches[test_mask], labels[test_mask], label_id[test_mask]


            mean = train_patches.mean(dim= (0,2,3)) # will average over the 0,2,3 axes and keep the 1 axis (channels / bands) separate. output shape (C,)
            std = train_patches.std(dim = (0,2,3))

            train_loader = load_dataset(patches=train_patches,
                                        labels= train_labels,
                                        mean= mean,
                                        std = std,
                                        batch_size= batch_size,
                                        shuffle= True)

            val_loader = load_dataset(patches = val_patches,
                                    labels= val_labels,
                                    mean= mean,
                                    std= std,
                                    shuffle = False,
                                    batch_size= batch_size
                                    )

            for epoch in range(num_epochs):

                train_loss, train_metrics = train_for_one_epoch(model, train_loader, device, aug, optimizer, metrics, criterion)
                val_loss, val_metrics = val_for_one_epoch(model, val_loader, metrics, device, criterion)

                # Save metrics for epoch to history dict
                record_epoch(history_dict= history_dict, run_ID=run_ID, patch_size = patch_size, loop = "inner", epoch=epoch, test_region=test_region, val_region=val_region,
                            train_loss=train_loss, val_loss= val_loss, train_metrics= train_metrics, val_metrics=val_metrics )

                scheduler.step()

                #print(f"Epoch {epoch+1}/100 | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_f1: {val_metrics['f1']:.4f}"
                 #   f"| val_acc: {val_metrics['accuracy']:.4f}")

                # Early stopping

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_train_loss = train_loss
                    best_val_metrics = val_metrics
                    best_train_metrics = train_metrics
                    best_epoch = epoch + 1 # Need to change to 1-based number, so can be recorded and calculate median for outer loop.
                    epochs_no_improve = 0
                else:
                    epochs_no_improve +=1
                    if epochs_no_improve >= patience:
                        print(f"Run ID: {run_ID}")
                        print(f"Early stopping at epcoch: {epoch+1}")
                        print(f"Best epoch train_loss: {best_train_loss} | Best val_loss: {best_val_loss} | Best epoch val_f1 {best_val_metrcis[f1]}")
                        break
            fold_histories.append(history_dict)


            inner_val_losses.append(best_val_loss)
            inner_f1_scores.append(best_val_metrics["f1"])
            inner_best_epochs.append(best_epoch)

            fold_metrics.append({
                "test_region": test_region,
                "val_region": val_region,
                "epoch_stopped": best_epoch,
                "train_loss": best_train_loss,
                "val_loss": best_val_loss,
                "val_metrics": best_val_metrics,
                "train_metrics": best_train_metrics
                })



        trial.set_user_attr("epochs", inner_best_epochs)
        trial.set_user_attr("fold_metrics", fold_metrics)
        trial.set_user_attr("fold_histories", fold_histories)

        return float(np.mean(inner_f1_scores)) # returns the mean f1 for each trial, previously had this as inner_valloss with create_stufy(ditection = "minimize")

    sampler = optuna.samplers.TPESampler(n_startup_trials=5, seed=42) # 5 trials with random params before tpe
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials = 15)

    best_params = study.best_trial.params # a dict of the parameters and the best values. Values from best trial chosed based on mean validation loss across fold in the trial (at best epoch)
    median_epoch = int(np.median(study.best_trial.user_attrs["epochs"])) # returns the median value of the epoch number at early stopping of the folds in the best trial.

    # Need a fresh best_state_dict for each outer fold, to save the per-fold metrics from the winning trial
    inner_best_state_dict= {"run_ID": [], "patch_size":[], "test_region": [], "val_region": [], "train_loss": [], "val_loss": [], "epoch_stopped": [],
                  "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": [], "val_auroc":[],"train_accuracy": [], "train_f1": [], "train_precision": [], "train_recall": [], "train_auroc": [],
    "dropout": [], "lr": [], "weight_decay":[], "batch_size": []}

    for record in study.best_trial.user_attrs["fold_metrics"]: # loops through the folds in the best_trial
        val_abrv = record["val_region"][0:3]
        run_ID = f"T-{test_abrv}_V-{val_abrv}_{patch_size}px_{date}"
        record_inner_loop_best_state(inner_best_state_dict= inner_best_state_dict,
                                     run_ID = run_ID,
                                     patch_size= patch_size,
                                     best_epoch= record["epoch_stopped"],
                                     test_region= test_region,
                                     val_region= record["val_region"],
                                     train_loss= record["train_loss"],
                                     best_val_loss= record["val_loss"],
                                     best_val_metrics= record["val_metrics"],
                                     best_train_metrics= record["train_metrics"],
                                     lr = best_params["lr"],
                                     dropout = best_params["dropout"],
                                     weight_decay = best_params["weight_decay"],
                                     batch_size = batch_size
                                     )


    list_of_inner_best_state_metrics.append(inner_best_state_dict) # save the best trial inner fold metrics to a list that can be concatenated later


    # plot the loss curves for the best trial fold, append to best trial list for later concatenation and saving

    best = study.best_trial
    for fold, record in zip(best.user_attrs["fold_histories"], best.user_attrs["fold_metrics"]):
        run_ID = fold["run_ID"][0] # run id is broadcast down the column for each epoch
        loss_curve_fp = OUTPUT_ROOT / "outputs/plots/loss_curves" / f"{run_ID}.png"
        plot_loss_curve(fold, loss_curve_fp)

        list_of_run_histories.append(fold)


        print(run_ID)
        print(f"Test region: {test_region}, Val region: {record['val_region']}")
        print(f"Best epoch: {record['epoch_stopped']} | val_loss: {record['val_loss']:.4f} | train_f1: {record['train_metrics']['f1']:.4f} val_f1: {record['val_metrics']['f1']:.4f}" )

    # Append this run's history dict to list for later concatenation.



    # -------------------------
    # Outer fold -> train on all remaining regions using median number of epochs from inner fold
    #---------------------------------------------------------------------------------------------
    history_dict = {"run_ID": [], "test_region": [], "patch_size":[], "loop":[], "epoch":[], "train_loss": [], "train_accuracy":[],"train_f1":[], "train_precision":[], "train_recall":[], "train_auroc":[]
                        }

    test_mask = locations == test_region
    remaining_mask = locations != test_region

    run_ID = f"Test_{test_abrv}_outer_{patch_size}px_{date}"

    remaining_patches = patches[remaining_mask]
    remaining_mean = remaining_patches.mean(dim= (0,2,3))
    remaining_std =  remaining_patches.std(dim= (0,2,3))

    test_patches = patches[test_mask]

    remaining_labels = labels[remaining_mask]
    test_labels = labels[test_mask]
    test_label_ids = label_id[test_mask]
    test_class_int = class_int[test_mask]

    remaining_loader = load_dataset(patches=remaining_patches,
                                    labels= remaining_labels,
                                    mean= remaining_mean,
                                    std= remaining_std,
                                    shuffle = True,
                                    batch_size = batch_size
                                    )


    test_loader = load_dataset(patches=test_patches,
                               labels = test_labels,
                               mean = remaining_mean,
                               std = remaining_std,
                               shuffle= False,
                               batch_size= batch_size
                               )

    model = buildCNN_2xVGG(input_channels= input_channels, dropout = best_params["dropout"])
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr = best_params["lr"], weight_decay = best_params["weight_decay"])

    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    for epoch in range(median_epoch):
        train_loss, train_metrics = train_for_one_epoch(model, remaining_loader, device, aug, optimizer, metrics, criterion)

        scheduler.step()

        print(f"Epoch {epoch+1}/100 | train_loss: {train_loss:.4f}")

        # save the per epoch metrics so can plot loss curve later

        record_epoch(history_dict= history_dict, run_ID=run_ID, patch_size = patch_size, loop = "outer", epoch=epoch, test_region=test_region,
                         train_loss=train_loss, train_metrics= train_metrics)

        loss_curve_fp = OUTPUT_ROOT / "outputs/plots/loss_curves" / f"{run_ID}.png"

    # Append trainining history dict
    list_of_run_histories.append(history_dict)

    # Use model to predict on the held-out test region, save preds for creating confusion matrix.

    test_loss, test_metrics, preds_tensor, logits = predict_on_test_region(model, test_loader, metrics, device, criterion)

    loss_curve_fp = OUTPUT_ROOT / "outputs/plots/loss_curves" / f"{run_ID}.png"

    plot_loss_curve(history_dict, loss_curve_fp)

    print(f"Predicting on held out test region: {test_region}")
    print(f"Trained for {median_epoch} epochs | Train loss: {train_loss} | Test loss {test_loss} | Test f1: {test_metrics["f1"]}")

    record_outer_loop_metrics(outer_metrics_dict, run_ID, patch_size, test_region, median_epoch, test_loss, train_loss, test_metrics, train_metrics, best_params["lr"],
                                    best_params["dropout"],best_params["weight_decay"], batch_size)


# Need to put together the preds with labels etc

    preds_array = preds_tensor.detach().squeeze(1).cpu().numpy()
    labels_array = test_labels.detach().squeeze(1).numpy()

    prediction_df = pd.DataFrame({
        "test_region": test_region,
        "run_id": run_ID,
        "label_id": test_label_ids,
        "labels": labels_array,
        "predicted": preds_array,
        "class_int": test_class_int
    })

    list_of_predicition_dfs.append(prediction_df)

    # save trained models and the relevant parameters for each outer fold?

    checkpoint = {
        "run_ID": run_ID,
        "test_region": test_region,
        "model_state_dict": model.state_dict(),
        "mean": remaining_mean,
        "std": remaining_std,
        "band_list": band_list,
        "median_epoch": median_epoch,
        "hyperparams": {"lr": best_params["lr"], "weight_decay": best_params["weight_decay"], "dropout": best_params["dropout"], "batch_size": batch_size},
    }
    model_fp = OUTPUT_ROOT / "outputs" / "CNN_outputs" / "models" / f"{run_ID}.pt"
    model_fp.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, model_fp)

#----------------------------------------------------------------------------------
# After loops
# Concat the the prediciton_dfs

predictions = pd.concat(list_of_predicition_dfs,axis=0)
predictions_fp = = OUTPUT_ROOT / "outputs" / "CNN_outputs" / f"Outer_fold_predictions_{patch_size}px_{date}.csv"

predictions.to_csv(predictions_fp)

# Need to combine the history dicts into a data frame
history_df = pd.concat([pd.DataFrame(h) for h in list_of_run_histories], ignore_index= True)

run_history_fp = OUTPUT_ROOT / "outputs" / "CNN_outputs" / f"training_run_history_{patch_size}px_{date}.csv"
history_df.to_csv(run_history_fp)

# save the outer metrics as data frame

outer_metrics_df =pd.DataFrame(outer_metrics_dict)

outer_metrics_fp = OUTPUT_ROOT / "outputs" / "CNN_outputs" / f"Outer_fold_metrics_{patch_size}px_{date}.csv"

outer_metrics_df.to_csv(outer_metrics_fp)
# save the best inner metrics as dataframe for each fold

inner_metrics_df = pd.concat([pd.DataFrame(h) for h in list_of_inner_best_state_metrics], ignore_index= True)

inner_metrics_fp = OUTPUT_ROOT / "outputs" / "CNN_outputs" / f"Inner_fold_best_state_metrics_{patch_size}px_{date}.csv"
inner_metrics_df.to_csv(inner_metrics_fp)



SyntaxError: invalid syntax (569111636.py, line 309)

In [ ]:
print(list_of_run_histories[0].keys())

dict_keys(['test_region', 'val_region', 'run_ID', 'patch_size', 'loop', 'epoch', 'train_loss', 'val_loss', 'val_accuracy', 'val_f1', 'val_precision', 'val_recall', 'val_auroc', 'train_accuracy', 'train_f1', 'train_precision', 'train_recall', 'train_auroc'])


In [ ]:
df = pd.DataFrame(list_of_run_histories[0])

In [ ]:
df.head()

,test_region,val_region,run_ID,patch_size,loop,epoch,train_loss,val_loss,val_accuracy,val_f1,val_precision,val_recall,val_auroc,train_accuracy,train_f1,train_precision,train_recall,train_auroc
0,Hartbeespoort,Rodman,T-Har_V-Rod_15px_2026-07-16,15,inner,1,0.295784,0.481665,0.832517,0.844687,0.788136,0.909980,0.894271,0.879488,0.878272,0.882688,0.873901,0.946921
1,Hartbeespoort,Rodman,T-Har_V-Rod_15px_2026-07-16,15,inner,2,0.189160,0.362758,0.866308,0.866242,0.867517,0.864971,0.944063,0.932315,0.932029,0.931257,0.932802,0.976979
